06 - Pack the corpus into a single archive
#
# CPU only, so it costs no GPU quota. Attaches the preprocess kernel and
# writes one tar, because kernels_output fetches loose files serially and
# 17k of them is unusable over the API.
#
# Uncompressed on purpose: the mosaics are already JPEG, so tar.gz would
# spend CPU to save almost nothing.

In [ ]:
import sys, subprocess
from pathlib import Path

CODE = next(
    (p for pat in ("rsna-knee-code", "*/rsna-knee-code", "*/*/rsna-knee-code")
     for p in sorted(Path("/kaggle/input").glob(pat)) if (p / "src").is_dir()),
    None,
)
assert CODE is not None, "rsna-knee-code dataset is not attached to this notebook"
sys.path.insert(0, str(CODE / "src"))
sys.path.insert(0, str(CODE))
print("CODE", CODE)

# timm ships in the Kaggle image; the install is a fallback for images that lack
# it, and cannot run in the offline submission notebook.
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm

import rsna_knee.config as cfg
print("RAW ", cfg.RAW, cfg.RAW.exists())
print("PREP", cfg.PREP, cfg.PREP.exists())
print("SIZE", cfg.SIZE, cfg.MOSAIC_EXT)

In [ ]:
import tarfile, time
from pathlib import Path

src = cfg.PREP
assert src.exists(), f"corpus not found at {src}"
files = sorted(p for p in src.rglob("*") if p.suffix.lower() in (".jpg", ".png"))
print(f"{len(files)} mosaics, {sum(p.stat().st_size for p in files)/1e9:.2f} GB")

out = Path("/kaggle/working/prep.tar")
t = time.time()
with tarfile.open(out, "w") as tar:
    for i, p in enumerate(files, 1):
        tar.add(p, arcname=str(p.relative_to(src.parent)))
        if i % 2000 == 0:
            print(f"  {i}/{len(files)}", flush=True)
print(f"wrote {out}  {out.stat().st_size/1e9:.2f} GB in {(time.time()-t)/60:.1f} min")